# Supply Chain Analytics - Exploratory Data Analysis (EDA)

## Objective

This notebook focuses on exploratory analysis and business insights generation from the processed supply chain datasets.

The goal is to answer key business questions:

- Which products and categories drive revenue and profit?
- Where is inventory risk concentrated?
- Which suppliers require attention?
- How efficiently are warehouses operating?
- How reliable are demand forecasts?

The analysis will support:
- Executive reporting
- Inventory optimization
- Supplier improvement
- Operational decision-making
- Tableau dashboard development

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs("processed_data", exist_ok=True)
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns",None)
pd.set_option("display.float_format",lambda x: f"{x:,.2f}")
print("Libraries loaded successfully")

Libraries loaded successfully


# Load Analytical Datasets

In [ ]:
sales_orders = pd.read_csv("sales_orders.csv")

inventory_dashboard = pd.read_csv("inventory_dashboard.csv")

warehouse_dashboard = pd.read_csv("warehouse_dashboard.csv")

supplier_dashboard = pd.read_csv("supplier_dashboard.csv")

forecast_dashboard = pd.read_csv("forecast_dashboard.csv")

warehouse_analysis = pd.read_csv("warehouse_analysis.csv")

print("Datasets Loaded")

Datasets Loaded


In [ ]:
# Dataset Shapes

datasets = {"sales orders": sales_orders, "inventory": inventory_dashboard,
            "warehouse": warehouse_dashboard,"supplier": supplier_dashboard,
            "forecast": forecast_dashboard}

for name, df in datasets.items():
    print("="*60)
    print(name)
    print("Shape:", df.shape)
    print("="*60)

sales orders
Shape: (292909, 20)
inventory
Shape: (25000, 54)
warehouse
Shape: (5, 18)
supplier
Shape: (200, 29)
forecast
Shape: (250, 14)


In [ ]:
# Data Type Validation

for name, df in datasets.items():
    print("\n", "="*60)
    print(name)
    print(df.dtypes)


sales orders
order_id               object
order_date             object
customer_id            object
product_id             object
warehouse_id           object
quantity                int64
selling_price         float64
discount_pct          float64
revenue               float64
cogs                  float64
profit                float64
order_status           object
shipping_method        object
delivery_time_days      int64
returned               object
customer_state         object
calculated_revenue    float64
expected_revenue      float64
net_revenue           float64
net_profit            float64
dtype: object

inventory
snapshot_date                   object
warehouse_id                    object
product_id                      object
opening_stock                    int64
received_qty                     int64
sold_qty                         int64
adjustment_qty                   int64
closing_stock                    int64
inventory_value                float64
days_of_in

In [ ]:
# Missing Value Check

for name, df in datasets.items():
    print("\n", "="*60)
    print(name)
    missing = (df.isnull().sum().sort_values(ascending=False))
    print(missing[missing>0])


sales orders
Series([], dtype: int64)

inventory
Series([], dtype: int64)

warehouse
Series([], dtype: int64)

supplier
Series([], dtype: int64)

forecast
Series([], dtype: int64)


In [ ]:
# Duplicate Check

for name, df in datasets.items():
    print("\n", "="*60)
    print(name)
    print("Duplicate Rows:",df.duplicated().sum())


sales orders
Duplicate Rows: 0

inventory
Duplicate Rows: 0

warehouse
Duplicate Rows: 0

supplier
Duplicate Rows: 0

forecast
Duplicate Rows: 0


In [ ]:
sales_orders["order_date"] = pd.to_datetime(sales_orders["order_date"])

inventory_dashboard["snapshot_date"] = pd.to_datetime(
    inventory_dashboard["snapshot_date"]
)

print("Date conversion completed.")

Date conversion completed.


In [ ]:
sales_orders_eda = sales_orders.drop_duplicates().copy()

print("Original :", sales_orders.shape)
print("EDA Copy :", sales_orders_eda.shape)

Original : (292909, 20)
EDA Copy : (292909, 20)


# Sales Performance Analysis

**Objective**

This section analyzes sales performance across products, customers, geography, and time.

**Business Questions**

- How much revenue and profit did the business generate?
- What is the overall profitability?
- How have sales changed over time?
- Which categories contribute the most revenue?
- Which states generate the highest sales?
- How do discounts impact profitability?

The findings from this section will support executive decision-making and revenue optimization.

In [ ]:
# Executive Sales KPIs

sales_kpis = pd.DataFrame({"Metric":["Total Revenue","Total Profit","Total Orders",
                                     "Units Sold","Average Order Value","Average Profit Margin (%)"],
                           "Value":
                            [sales_orders_eda["revenue"].sum(),sales_orders_eda["profit"].sum(),
                             sales_orders_eda["order_id"].nunique(),
                             sales_orders_eda["quantity"].sum(),
                             sales_orders_eda["revenue"].sum() / sales_orders_eda["order_id"].nunique(),
                              (sales_orders_eda["profit"].sum() / sales_orders_eda["revenue"].sum())*100
                             ]
                           })

sales_kpis

,Metric,Value
0,Total Revenue,"37,587,376.31"
1,Total Profit,"8,219,595.94"
2,Total Orders,"292,909.00"
3,Units Sold,"454,526.00"
4,Average Order Value,128.32
5,Average Profit Margin (%),21.87


In [ ]:
# =====================================================
# Monthly Sales Trend
# =====================================================

sales_orders_eda["year_month"] = (

    sales_orders_eda["order_date"]

    .dt.to_period("M")

    .astype(str)

)

monthly_sales = (

    sales_orders_eda

    .groupby("year_month")

    .agg(

        revenue=("revenue","sum"),

        profit=("profit","sum"),

        orders=("order_id","nunique"),

        units=("quantity","sum")

    )

    .reset_index()

)

monthly_sales.head()

,year_month,revenue,profit,orders,units
0,2023-01,"1,472,980.66","326,823.09",11220,17251
1,2023-02,"1,320,518.59","297,431.42",10200,15912
2,2023-03,"1,431,021.88","322,683.98",11305,17601
3,2023-04,"1,454,040.16","330,955.43",10965,16992
4,2023-05,"1,472,329.96","332,490.54",11220,17457


## Category Performance

In [ ]:
sales_category = (

    sales_orders_eda

    .merge(

        inventory_dashboard[
            [
                "product_id",
                "product_name",
                "category",
                "subcategory",
                "brand"
            ]
        ],

        on="product_id",

        how="left"

    )

)

In [ ]:
category_sales = (

    sales_category

    .groupby("category")

    .agg(

        revenue=("revenue","sum"),

        profit=("profit","sum"),

        orders=("order_id","nunique"),

        units=("quantity","sum")

    )

    .sort_values(

        "revenue",

        ascending=False

    )

)

category_sales

,revenue,profit,orders,units
category,,,,
Electronics,"78,310,127.55","11,271,129.75",37308,186540
Furniture,"45,455,343.85","13,583,174.25",38938,194690
Home & Kitchen,"13,808,189.15","3,451,198.45",37704,282780
Office Supplies,"12,396,504.25","3,162,581.80",35876,269195
Fashion,"12,206,128.30","3,065,930.90",36158,271255
Beauty,"12,142,402.65","3,078,517.45",37348,280420
Sports,"10,326,336.45","2,563,714.55",33942,254150
Grocery,"3,291,849.35","921,732.55",35635,533600


## Shipping Performance

In [ ]:
shipping_analysis = (

    sales_orders_eda

    .groupby("shipping_method")

    .agg(

        revenue=("revenue","sum"),

        avg_delivery_days=(

            "delivery_time_days",

            "mean"

        ),

        orders=(

            "order_id",

            "count"

        )

    )

    .sort_values(

        "revenue",

        ascending=False

    )

)

shipping_analysis

,revenue,avg_delivery_days,orders
shipping_method,,,
Standard Ground,"18,832,130.13",3.90,146745
Expedited,"9,377,161.04",3.90,73016
Two-Day Air,"5,654,594.65",3.90,43931
Overnight Saver,"3,723,490.49",3.90,29217


## Customer Geography

In [ ]:
state_sales = (

    sales_orders_eda

    .groupby("customer_state")

    .agg(

        revenue=("revenue","sum"),

        profit=("profit","sum"),

        orders=("order_id","count")

    )

    .sort_values(

        "revenue",

        ascending=False

    )

)

state_sales.head(10)

,revenue,profit,orders
customer_state,,,
GA,"1,994,855.99","431,837.72",14903
FL,"1,901,185.45","415,469.06",14794
MA,"1,896,490.93","415,203.36",14743
OH,"1,893,944.63","419,882.36",14751
TX,"1,892,853.75","410,696.81",14612
WI,"1,889,937.08","411,738.62",14625
AZ,"1,885,477.57","415,237.89",14721
TN,"1,885,255.75","409,682.97",14765
NC,"1,883,877.87","406,190.56",14571


## Discount Analysis

In [ ]:
sales_orders_eda["discount_band"] = pd.cut(

    sales_orders_eda["discount_pct"],

    bins=[

        -1,
        0,
        10,
        20,
        30,
        100

    ],

    labels=[

        "No Discount",

        "1-10%",

        "11-20%",

        "21-30%",

        "30%+"

    ]

)

In [ ]:
discount_analysis = (

    sales_orders_eda

    .groupby("discount_band")

    .agg(

        revenue=("revenue","sum"),

        profit=("profit","sum"),

        orders=("order_id","count"),

        avg_profit=("profit","mean")

    )

)

discount_analysis

,revenue,profit,orders,avg_profit
discount_band,,,,
No Discount,"30,818,010.27","7,337,751.36",234401,31.30
1-10%,"3,772,183.44","660,093.36",31004,21.29
11-20%,"2,516,821.26","229,053.65",22682,10.10
21-30%,"480,361.34","-7,302.43",4822,-1.51
30%+,0.00,0.00,0,NaN


## Top Products

In [ ]:
top_products = (

    sales_category

    .groupby(

        [

            "product_id",

            "product_name"

        ]

    )

    .agg(

        revenue=("revenue","sum"),

        profit=("profit","sum"),

        units=("quantity","sum")

    )

    .sort_values(

        "revenue",

        ascending=False

    )

    .head(20)

)

top_products

,,revenue,profit,units
product_id,product_name,,,
PROD02004,Smartphones Model C2004,"1,060,825.20","166,466.85",785
PROD04174,Accessories Model O4174,"1,056,795.45","192,480.45",750
PROD03884,Tablets Model K3884,"980,804.15","112,759.95",745
PROD04135,Laptops Model B4135,"944,957.55","166,166.25",735
PROD01185,Accessories Model P1185,"933,354.65","168,578.05",665
PROD00566,Smartphones Model U566,"928,650.85","174,554.65",660
PROD01036,Cameras Model W1036,"885,937.50","137,063.90",760
PROD04078,Tablets Model W4078,"881,674.60","121,113.35",725
PROD02239,Cameras Model D2239,"877,889.80","128,475.40",640


In [ ]:
# =====================================================
# Saving Enriched Sales Dataset
# =====================================================

sales_category.to_csv(
    "processed_data/sales_category_analysis.csv",
    index=False
)

print("Sales category analysis dataset saved.")

Sales category analysis dataset saved.


# Product & Inventory Analysis

***Objective***

Analyze product-level inventory performance to identify:

- High-value products
- Inventory risks
- Slow-moving stock
- Overstock situations
- Replenishment opportunities

The analysis supports inventory optimization decisions.

In [ ]:
# =====================================================
# Inventory KPIs
# =====================================================

inventory_kpis = pd.DataFrame({"Metric":
 ["Total Inventory Value","Total Units Available","Total SKUs",
  "Average Days of Inventory","Average Inventory Turnover"],
                               "Value":[
                                   inventory_dashboard["inventory_value"].sum(),
                                   inventory_dashboard["closing_stock"].sum(),
                                   inventory_dashboard["product_id"].nunique(),
                                   inventory_dashboard["days_of_inventory"].mean(),
                                   inventory_dashboard["inventory_turnover"].mean()
                                   ]
                               })

inventory_kpis

,Metric,Value
0,Total Inventory Value,"277,176,380.63"
1,Total Units Available,"3,521,433.00"
2,Total SKUs,"5,000.00"
3,Average Days of Inventory,100.39
4,Average Inventory Turnover,6.26


In [ ]:
# =====================================================
# Inventory Health Distribution
# =====================================================

inventory_status = (

    inventory_dashboard

    .groupby("inventory_status")

    .agg(

        inventory_records=("product_id","count"),

        inventory_value=("inventory_value","sum"),

        units=("closing_stock","sum")

    )

    .sort_values(

        "inventory_value",

        ascending=False

    )

)


inventory_status

,inventory_records,inventory_value,units
inventory_status,,,
Healthy,16224,"201,246,800.66",2571435
Overstock,2536,"48,441,826.49",646278
Reorder Required,3774,"22,044,212.46",252132
Stockout Risk,2466,"5,443,541.02",51588


In [ ]:
# =====================================================
# Inventory Velocity
# =====================================================

velocity_analysis = (

    inventory_dashboard

    .groupby("inventory_velocity")

    .agg(

        sku_count=("product_id","nunique"),

        inventory_value=("inventory_value","sum"),

        avg_days_inventory=("days_of_inventory","mean")

    )

    .sort_values(

        "inventory_value",

        ascending=False

    )

)


velocity_analysis

,sku_count,inventory_value,avg_days_inventory
inventory_velocity,,,
Normal Moving,2050,"188,489,788.61",59.57
Slow Moving,2238,"65,290,765.04",126.30
Fast Moving,207,"15,813,810.59",31.09
Dead Stock,505,"7,582,016.39",179.69


In [ ]:
# =====================================================
# Top Products by Inventory Value
# =====================================================

top_inventory_products = (

    inventory_dashboard

    .groupby(

        [

            "product_id",

            "product_name"

        ]

    )

    .agg(

        inventory_value=("inventory_value","sum"),

        current_stock=("closing_stock","sum"),

        turnover=("inventory_turnover","mean"),

        inventory_status=("inventory_status","first")

    )

    .sort_values(

        "inventory_value",

        ascending=False

    )

    .head(20)

)


top_inventory_products

,,inventory_value,current_stock,turnover,inventory_status
product_id,product_name,,,,
PROD02239,Cameras Model D2239,"2,679,156.48",2288,0.33,Healthy
PROD03884,Tablets Model K3884,"2,417,707.00",2075,0.30,Healthy
PROD01504,Accessories Model W1504,"2,162,213.40",2220,0.22,Healthy
PROD02252,Cameras Model Q2252,"2,112,158.37",2261,0.30,Reorder Required
PROD01065,Cameras Model Z1065,"2,106,917.40",1811,0.45,Healthy
PROD02580,Accessories Model G2580,"2,087,075.89",1927,0.38,Overstock
PROD00566,Smartphones Model U566,"1,983,501.52",1736,0.38,Healthy
PROD02288,Laptops Model A2288,"1,894,480.08",3144,0.26,Healthy
PROD02717,Smartphones Model N2717,"1,855,196.10",1602,0.33,Healthy


In [ ]:
# =====================================================
# Dead Stock Identification
# =====================================================

dead_stock = (

    inventory_dashboard[

        inventory_dashboard["inventory_velocity"]

        ==

        "Dead Stock"

    ]

    .groupby(

        [

            "product_id",

            "product_name"

        ]

    )

    .agg(

        inventory_value=("inventory_value","sum"),

        days_inventory=("days_of_inventory","mean")

    )

    .sort_values(

        "inventory_value",

        ascending=False

    )

)


dead_stock.head(20)

,,inventory_value,days_inventory
product_id,product_name,,
PROD02697,Laptops Model T2697,"133,956.30",313.82
PROD02145,Laptops Model N2145,"103,250.38",229.30
PROD01623,Tablets Model L1623,"101,144.59",215.98
PROD01030,Cameras Model Q1030,"100,809.67",245.50
PROD03382,Accessories Model C3382,"98,810.12",345.50
PROD04038,Tablets Model I4038,"90,601.40",222.00
PROD01331,Tablets Model F1331,"85,289.68",185.22
PROD01897,Cameras Model Z1897,"84,813.12",197.42
PROD01208,Smartphones Model M1208,"81,639.66",251.12


In [ ]:
# =====================================================
# Category Inventory Analysis
# =====================================================

category_inventory = (

    inventory_dashboard

    .groupby("category")

    .agg(

        inventory_value=("inventory_value","sum"),

        units=("closing_stock","sum"),

        avg_days_inventory=("days_of_inventory","mean"),

        sku_count=("product_id","nunique")

    )

    .sort_values(

        "inventory_value",

        ascending=False

    )

)


category_inventory

,inventory_value,units,avg_days_inventory,sku_count
category,,,,
Electronics,"139,488,737.58",445256,98.81,633
Furniture,"70,661,881.94",477174,103.69,650
Home & Kitchen,"14,288,859.13",440088,99.10,613
Fashion,"14,131,231.95",453374,101.53,626
Office Supplies,"13,178,902.44",432326,102.39,605
Beauty,"12,495,995.01",433452,96.83,640
Sports,"11,030,583.44",413169,103.00,626
Grocery,"1,900,189.14",426594,97.69,607


# Supplier Performance Analysis

## Objective

Evaluate supplier performance using operational metrics:

- Delivery reliability
- Defect rate
- Fill rate
- Delay performance
- Overall supplier score
- Supplier risk classification

The analysis helps identify strategic suppliers and improvement opportunities.

In [ ]:
# =====================================================
# Supplier KPIs
# =====================================================

supplier_kpis = pd.DataFrame({

    "Metric":[

        "Total Suppliers",
        "Average Supplier Score",
        "Average On-Time Delivery %",
        "Average Defect Rate %",
        "Average Fill Rate"

    ],

    "Value":[

        supplier_dashboard["supplier_id"].nunique(),

        supplier_dashboard["supplier_performance_score"].mean(),

        supplier_dashboard["on_time_delivery_pct"].mean(),

        supplier_dashboard["defect_rate_pct"].mean(),

        supplier_dashboard["fill_rate"].mean()

    ]

})


supplier_kpis

,Metric,Value
0,Total Suppliers,200.00
1,Average Supplier Score,83.94
2,Average On-Time Delivery %,82.22
3,Average Defect Rate %,3.82
4,Average Fill Rate,94.58


In [ ]:
# =====================================================
# Supplier Risk Analysis
# =====================================================

supplier_risk = (

    supplier_dashboard

    .groupby("supplier_risk")

    .agg(

        supplier_count=("supplier_id","count"),

        avg_score=("supplier_performance_score","mean"),

        avg_delivery=("on_time_delivery_pct","mean"),

        avg_defect_rate=("defect_rate_pct","mean")

    )

    .sort_values(

        "avg_score",

        ascending=False

    )

)


supplier_risk

,supplier_count,avg_score,avg_delivery,avg_defect_rate
supplier_risk,,,,
Low Risk,28,94.65,95.45,0.79
Medium Risk,88,83.41,83.06,3.65
High Risk,84,80.91,76.93,5.01


In [ ]:
# =====================================================
# Best Suppliers
# =====================================================

top_suppliers = (

    supplier_dashboard

    .sort_values(

        "supplier_performance_score",

        ascending=False

    )

    .head(20)

)


top_suppliers[
    [
        "supplier_id",
        "supplier_performance_score",
        "on_time_delivery_pct",
        "defect_rate_pct",
        "fill_rate",
        "supplier_risk"
    ]
]

,supplier_id,supplier_performance_score,on_time_delivery_pct,defect_rate_pct,fill_rate,supplier_risk
6,SUP0007,97.40,99.37,0.94,96.64,Low Risk
28,SUP0029,96.90,96.42,0.56,97.64,Low Risk
14,SUP0015,96.50,97.18,0.48,95.48,Medium Risk
26,SUP0027,96.20,94.80,0.71,97.75,Low Risk
21,SUP0022,96.00,96.44,1.24,98.30,Low Risk
18,SUP0019,95.90,98.43,0.44,98.64,Low Risk
9,SUP0010,95.80,94.65,0.97,97.42,Low Risk
3,SUP0004,95.70,96.86,0.17,96.85,Low Risk
5,SUP0006,95.70,93.66,0.70,99.20,Low Risk
10,SUP0011,95.40,92.96,0.76,97.58,Low Risk


In [ ]:
# =====================================================
# Supplier Improvement Candidates
# =====================================================

supplier_risk_priority = (

    supplier_dashboard

    .sort_values(

        [
            "supplier_performance_score",
            "defect_rate_pct"

        ],

        ascending=[
            True,
            False
        ]

    )

    .head(20)

)


supplier_risk_priority[
    [
        "supplier_id",
        "supplier_performance_score",
        "defect_rate_pct",
        "on_time_delivery_pct",
        "supplier_risk"
    ]
]

,supplier_id,supplier_performance_score,defect_rate_pct,on_time_delivery_pct,supplier_risk
119,SUP0120,74.20,1.91,71.94,High Risk
168,SUP0169,75.40,4.81,70.13,High Risk
60,SUP0061,75.80,1.68,71.65,High Risk
67,SUP0068,76.10,2.36,72.80,High Risk
96,SUP0097,76.20,4.05,71.87,High Risk
151,SUP0152,76.20,3.98,74.83,High Risk
103,SUP0104,76.30,6.50,77.56,High Risk
116,SUP0117,76.40,6.49,70.81,High Risk
156,SUP0157,76.50,5.45,76.84,High Risk
121,SUP0122,76.60,1.85,70.39,High Risk


In [ ]:
# =====================================================
# Delivery Status Analysis
# =====================================================

delivery_analysis = (

    supplier_dashboard

    .groupby("delivery_status")

    .agg(

        suppliers=("supplier_id","count"),

        avg_score=("supplier_performance_score","mean"),

        avg_delay=("avg_delay_days","mean"),

        avg_fill_rate=("fill_rate","mean")

    )

)


delivery_analysis

,suppliers,avg_score,avg_delay,avg_fill_rate
delivery_status,,,,
Average,59,84.56,0.27,95.05
Excellent,5,95.22,0.36,98.30
Good,22,94.75,0.21,97.24
Poor,114,81.03,0.23,93.65


In [ ]:
supplier_ranking = (

    supplier_dashboard

    .assign(
        rank=lambda x:
        x["supplier_performance_score"]
        .rank(
            ascending=False,
            method="dense"
        )
    )

    [
        [
            "rank",
            "supplier_id",
            "supplier_performance_score",
            "supplier_risk",
            "delivery_status"
        ]
    ]

    .sort_values("rank")

)


supplier_ranking.head(20)

,rank,supplier_id,supplier_performance_score,supplier_risk,delivery_status
6,1.00,SUP0007,97.40,Low Risk,Good
28,2.00,SUP0029,96.90,Low Risk,Good
14,3.00,SUP0015,96.50,Medium Risk,Average
26,4.00,SUP0027,96.20,Low Risk,Good
21,5.00,SUP0022,96.00,Low Risk,Excellent
18,6.00,SUP0019,95.90,Low Risk,Excellent
9,7.00,SUP0010,95.80,Low Risk,Good
3,8.00,SUP0004,95.70,Low Risk,Good
5,8.00,SUP0006,95.70,Low Risk,Good
10,9.00,SUP0011,95.40,Low Risk,Good


In [ ]:
supplier_risk_matrix = (

    supplier_dashboard

    .assign(

        performance_group=pd.cut(
            supplier_dashboard["supplier_performance_score"],
            bins=[0,80,90,100],
            labels=[
                "Low Performance",
                "Average Performance",
                "High Performance"
            ]
        )

    )

    .groupby(
        [
            "performance_group",
            "supplier_risk"
        ]
    )

    .agg(
        suppliers=("supplier_id","count"),
        avg_score=("supplier_performance_score","mean"),
        avg_delivery=("on_time_delivery_pct","mean"),
        avg_defect=("defect_rate_pct","mean")
    )

)

supplier_risk_matrix

suppliers  avg_score  avg_delivery  \
performance_group   supplier_risk                                       
Low Performance     High Risk             36      77.79         73.57   
                    Low Risk               0        NaN           NaN   
                    Medium Risk           12      79.28         78.71   
Average Performance High Risk             48      83.25         79.44   
                    Low Risk               1      89.60         88.71   
                    Medium Risk           73      83.62         83.19   
High Performance    High Risk              0        NaN           NaN   
                    Low Risk              27      94.83         95.70   
                    Medium Risk            3      95.00         97.42   

                                   avg_defect  
performance_group   supplier_risk              
Low Performance     High Risk            4.81  
                    Low Risk              NaN  
                    Medium Risk          4.24  
Average Performance High Risk            5.16  
                    Low Risk             2.67  
                    Medium Risk          3.68  
High Performance    High Risk             NaN  
                    Low Risk             0.72  
                    Medium Risk          0.71

# Business interpretation:

1. The supplier network maintains a strong fulfillment capability with a 94.58% average fill rate, but delivery reliability remains an improvement area with average on-time delivery of 82.22%.

2. 86% of suppliers require monitoring or improvement initiatives.

3. Identify low-risk suppliers as preferred vendors and evaluate opportunities to increase their allocation.

In [ ]:
supplier_risk_matrix.to_csv(
    "processed_data/supplier_risk_matrix.csv",
    index=True
)

# 5. Warehouse Performance Analysis

## Objective

Evaluate warehouse-level operational performance using:

- Inventory value
- Stock levels
- Capacity utilization
- Inventory health
- Revenue contribution
- Profit contribution

The goal is to identify warehouse efficiency gaps and optimization opportunities.

In [ ]:
# =====================================================
# Warehouse KPI Overview
# =====================================================

warehouse_kpis = pd.DataFrame({

    "Metric":[

        "Total Warehouses",
        "Total Inventory Value",
        "Average Capacity Utilization %",
        "Average Inventory Days",
        "Total Revenue",
        "Total Profit"

    ],

    "Value":[

        warehouse_dashboard["warehouse_id"].nunique(),

        warehouse_dashboard["total_inventory_value"].sum(),

        warehouse_dashboard["capacity_utilization_pct"].mean(),

        warehouse_dashboard["avg_days_inventory"].mean(),

        warehouse_dashboard["revenue"].sum(),

        warehouse_dashboard["profit"].sum()

    ]

})


warehouse_kpis

,Metric,Value
0,Total Warehouses,5.00
1,Total Inventory Value,"277,176,380.63"
2,Average Capacity Utilization %,86.38
3,Average Inventory Days,100.39
4,Total Revenue,"37,958,704.08"
5,Total Profit,"8,300,442.84"


In [ ]:
# =====================================================
# Warehouse Inventory Summary
# =====================================================

warehouse_summary = (

    warehouse_dashboard

    [
        [
            "warehouse_id",
            "total_inventory_value",
            "total_units",
            "avg_days_inventory",
            "capacity_utilization_pct"
        ]
    ]

    .sort_values(
        "total_inventory_value",
        ascending=False
    )

)


warehouse_summary

,warehouse_id,total_inventory_value,total_units,avg_days_inventory,capacity_utilization_pct
0,WH001,"56,858,795.64",707511,100.12,86.77
2,WH003,"55,666,791.53",704054,100.50,86.35
4,WH005,"55,254,125.20",700543,101.17,85.92
3,WH004,"54,951,771.92",708996,100.47,86.96
1,WH002,"54,444,896.34",700329,99.69,85.89


In [ ]:
# =====================================================
# Warehouse Stock Health
# =====================================================

warehouse_stock_health = (

    inventory_dashboard

    .groupby(
        [
            "warehouse_id",
            "inventory_status"
        ]
    )

    .agg(

        sku_warehouse_count=(
            "product_id",
            "count"
        ),

        inventory_value=(
            "inventory_value",
            "sum"
        )

    )

    .reset_index()

)


warehouse_stock_health

,warehouse_id,inventory_status,sku_warehouse_count,inventory_value
0,WH001,Healthy,3253,"42,805,708.69"
1,WH001,Overstock,494,"9,010,877.03"
2,WH001,Reorder Required,751,"3,984,831.89"
3,WH001,Stockout Risk,502,"1,057,378.03"
4,WH002,Healthy,3194,"37,514,815.06"
5,WH002,Overstock,526,"11,298,679.29"
6,WH002,Reorder Required,778,"4,483,988.71"
7,WH002,Stockout Risk,502,"1,147,413.28"
8,WH003,Healthy,3272,"41,508,347.66"
9,WH003,Overstock,500,"9,158,501.32"


In [ ]:
# =====================================================
# Warehouse Ranking
# =====================================================

warehouse_ranking = (

    warehouse_dashboard

    [
        [
            "warehouse_id",
            "warehouse_efficiency_score",
            "capacity_utilization_pct",
            "revenue",
            "profit"
        ]
    ]

    .sort_values(

        "warehouse_efficiency_score",

        ascending=False

    )

)


warehouse_ranking

,warehouse_id,warehouse_efficiency_score,capacity_utilization_pct,revenue,profit
2,WH003,94.80,86.35,"7,566,223.52","1,651,677.62"
0,WH001,90.50,86.77,"7,621,216.76","1,670,695.99"
4,WH005,81.50,85.92,"7,570,600.18","1,656,904.34"
1,WH002,77.30,85.89,"7,629,208.49","1,669,208.21"
3,WH004,69.10,86.96,"7,571,455.13","1,651,956.68"


In [ ]:
# =====================================================
# Warehouse Business Contribution
# =====================================================

warehouse_business = (

    warehouse_dashboard

    .groupby("warehouse_id")

    .agg(

        revenue=(
            "revenue",
            "sum"
        ),

        profit=(
            "profit",
            "sum"
        ),

        orders=(
            "total_orders",
            "sum"
        ),

        units_sold=(
            "units_sold",
            "sum"
        )

    )

    .sort_values(
        "profit",
        ascending=False
    )

)


warehouse_business

,revenue,profit,orders,units_sold
warehouse_id,,,,
WH001,"7,621,216.76","1,670,695.99",59126,91659
WH002,"7,629,208.49","1,669,208.21",59289,92062
WH005,"7,570,600.18","1,656,904.34",59225,92097
WH004,"7,571,455.13","1,651,956.68",59357,92148
WH003,"7,566,223.52","1,651,677.62",58841,91101


In [ ]:
# =====================================================
# Capacity Classification
# =====================================================

warehouse_dashboard["capacity_status"] = np.select(

    [

        warehouse_dashboard["capacity_utilization_pct"] >= 90,

        warehouse_dashboard["capacity_utilization_pct"] >= 75

    ],

    [

        "High Utilization",

        "Moderate Utilization"

    ],

    default="Low Utilization"

)


warehouse_capacity = (

    warehouse_dashboard

    [
        [
            "warehouse_id",
            "capacity_utilization_pct",
            "capacity_status"
        ]
    ]

)


warehouse_capacity

,warehouse_id,capacity_utilization_pct,capacity_status
0,WH001,86.77,Moderate Utilization
1,WH002,85.89,Moderate Utilization
2,WH003,86.35,Moderate Utilization
3,WH004,86.96,Moderate Utilization
4,WH005,85.92,Moderate Utilization


In [ ]:
warehouse_dashboard["capacity_rank"] = (
    warehouse_dashboard["capacity_utilization_pct"]
    .rank(
        ascending=False,
        method="dense"
    )
)

warehouse_dashboard[
[
"warehouse_id",
"capacity_utilization_pct",
"capacity_rank"
]
]

,warehouse_id,capacity_utilization_pct,capacity_rank
0,WH001,86.77,2.00
1,WH002,85.89,5.00
2,WH003,86.35,3.00
3,WH004,86.96,1.00
4,WH005,85.92,4.00


**Interpretation**

- The warehouse network is operating at a moderately high utilization level (86.38%).

- This is healthy because:

  1.Capacity is being utilized efficiently.

  2.There is still ~14% buffer available.

  3.No warehouse is currently at immediate storage risk.


However:

If inventory grows significantly, WH004 and WH001 should be monitored because they are closest to capacity pressure.



- Inventory is evenly distributed across warehouses, reducing dependency risk on a single location.

- Warehouse location is not driving major revenue differences; operational efficiency is the main differentiator.

In [ ]:
warehouse_final_analysis = (

    warehouse_dashboard

    [
        [
            "warehouse_id",
            "total_inventory_value",
            "capacity_utilization_pct",
            "capacity_rank",
            "avg_days_inventory",
            "warehouse_efficiency_score",
            "warehouse_rank",
            "revenue",
            "profit"
        ]
    ]

    .sort_values(
        "warehouse_efficiency_score",
        ascending=False
    )

)


warehouse_final_analysis

,warehouse_id,total_inventory_value,capacity_utilization_pct,capacity_rank,avg_days_inventory,warehouse_efficiency_score,warehouse_rank,revenue,profit
2,WH003,"55,666,791.53",86.35,3.00,100.50,94.80,1,"7,566,223.52","1,651,677.62"
0,WH001,"56,858,795.64",86.77,2.00,100.12,90.50,2,"7,621,216.76","1,670,695.99"
4,WH005,"55,254,125.20",85.92,4.00,101.17,81.50,3,"7,570,600.18","1,656,904.34"
1,WH002,"54,444,896.34",85.89,5.00,99.69,77.30,4,"7,629,208.49","1,669,208.21"
3,WH004,"54,951,771.92",86.96,1.00,100.47,69.10,5,"7,571,455.13","1,651,956.68"


In [ ]:
warehouse_final_analysis.to_csv(
    "processed_data/warehouse_final_analysis.csv",
    index=False
)

# Forecast & Demand Analysis

## The objective here is to evaluate:

- How accurate our forecasts are
- Where forecasting errors occur
- How seasonality affects demand
- Which products have unstable demand
- Which products require forecasting attention

In [ ]:
forecast_kpis = pd.DataFrame({

    "Metric":[

        "Total Products",
        "Average Forecast Accuracy %",
        "Average Forecast Error %",
        "Average Forecast Quantity",
        "Average Actual Quantity",
        "Forecast Risk Products"

    ],

    "Value":[

        forecast_dashboard["product_id"].nunique(),

        forecast_dashboard["avg_forecast_accuracy"].mean(),

        forecast_dashboard["avg_forecast_error"].mean(),

        forecast_dashboard["avg_forecast_qty"].mean(),

        forecast_dashboard["avg_actual_qty"].mean(),

        forecast_dashboard[
            "forecast_risk"
        ].value_counts()
        .get("High Risk",0)

    ]

})


forecast_kpis

,Metric,Value
0,Total Products,250.00
1,Average Forecast Accuracy %,87.29
2,Average Forecast Error %,12.71
3,Average Forecast Quantity,35.33
4,Average Actual Quantity,35.74
5,Forecast Risk Products,47.00


In [ ]:
forecast_risk_summary = (

    forecast_dashboard

    .groupby("forecast_risk")

    .agg(

        products=(
            "product_id",
            "count"
        ),

        avg_accuracy=(
            "avg_forecast_accuracy",
            "mean"
        ),

        avg_error=(
            "avg_forecast_error",
            "mean"
        )

    )

    .sort_values(
        "products",
        ascending=False
    )

)


forecast_risk_summary

,products,avg_accuracy,avg_error
forecast_risk,,,
Medium Risk,175,87.51,12.49
High Risk,47,85.20,14.80
Low Risk,28,89.42,10.58


In [ ]:
demand_category_analysis = (

    forecast_dashboard

    .groupby("demand_category")

    .agg(

        products=(
            "product_id",
            "count"
        ),

        avg_demand=(
            "avg_actual_qty",
            "mean"
        ),

        avg_variability=(
            "demand_variability",
            "mean"
        ),

        avg_accuracy=(
            "avg_forecast_accuracy",
            "mean"
        )

    )

)


demand_category_analysis

,products,avg_demand,avg_variability,avg_accuracy
demand_category,,,,
High Variability,41,7.21,0.57,86.20
Medium Variability,85,22.25,0.50,86.66
Stable Demand,124,54.43,0.36,88.08


In [ ]:
best_forecast_products = (

    forecast_dashboard

    [
        [
            "product_id",
            "category",
            "avg_forecast_accuracy",
            "forecast_risk"
        ]
    ]

    .sort_values(
        "avg_forecast_accuracy",
        ascending=False
    )

    .head(20)

)


best_forecast_products

,product_id,category,avg_forecast_accuracy,forecast_risk
135,PROD02697,Electronics,90.65,Low Risk
72,PROD01322,Grocery,90.43,Low Risk
130,PROD02587,Grocery,89.87,Low Risk
127,PROD02528,Sports,89.78,Low Risk
37,PROD00656,Furniture,89.65,Low Risk
198,PROD03863,Electronics,89.64,Low Risk
44,PROD00764,Fashion,89.60,Low Risk
216,PROD04257,Grocery,89.60,Low Risk
108,PROD02132,Sports,89.57,Low Risk
188,PROD03685,Beauty,89.55,Low Risk


In [ ]:
worst_forecast_products = (

    forecast_dashboard

    [
        [
            "product_id",
            "category",
            "avg_forecast_accuracy",
            "forecast_risk"
        ]
    ]

    .sort_values(
        "avg_forecast_accuracy"
    )

    .head(20)

)


worst_forecast_products

,product_id,category,avg_forecast_accuracy,forecast_risk
41,PROD00712,Electronics,82.62,High Risk
161,PROD03162,Grocery,83.26,High Risk
42,PROD00734,Furniture,83.76,High Risk
163,PROD03231,Beauty,83.92,High Risk
172,PROD03419,Sports,84.07,High Risk
131,PROD02634,Home & Kitchen,84.24,High Risk
9,PROD00145,Home & Kitchen,84.32,High Risk
240,PROD04772,Home & Kitchen,84.37,High Risk
106,PROD02105,Office Supplies,84.58,High Risk
145,PROD02887,Grocery,84.74,High Risk


In [ ]:
category_forecast_analysis = (

    forecast_dashboard

    .groupby("category")

    .agg(

        products=(
            "product_id",
            "count"
        ),

        avg_accuracy=(
            "avg_forecast_accuracy",
            "mean"
        ),

        avg_error=(
            "avg_forecast_error",
            "mean"
        ),

        total_actual_demand=(
            "total_actual_qty",
            "sum"
        )

    )

    .sort_values(
        "total_actual_demand",
        ascending=False
    )

)


category_forecast_analysis

,products,avg_accuracy,avg_error,total_actual_demand
category,,,,
Office Supplies,31,87.15,12.85,160405
Home & Kitchen,33,87.17,12.83,129534
Beauty,29,87.46,12.54,126369
Electronics,31,87.54,12.46,118791
Sports,35,87.09,12.91,118753
Furniture,35,87.38,12.62,112589
Fashion,32,87.20,12.80,88012
Grocery,24,87.35,12.65,83774


# Cross Functional Supply Chain Insights

## Objective

Find products where multiple risks exist together:

1. High inventory holding
2. Poor forecast performance
3. Demand uncertainty

These are the products where management should take action first.

We will combine:

- Inventory data
- Forecast data

In [ ]:
# =====================================================
# Inventory + Forecast Risk Analysis
# =====================================================

inventory_forecast_risk = (

    inventory_dashboard

    .merge(

        forecast_dashboard,

        on="product_id",

        how="left"

    )

)


inventory_forecast_risk.shape

(25000, 67)

In [ ]:
# =====================================================
# Risk Priority Score
# =====================================================

inventory_forecast_risk["risk_priority_score"] = (

    (inventory_forecast_risk["inventory_status"]
        .isin(
            [
                "Overstock",
                "Stockout Risk"
            ]
        )
        .astype(int)
        * 3)

    +

    (inventory_forecast_risk["forecast_risk"]
        .isin(
            [
                "High Risk"
            ]
        )
        .astype(int)
        * 2)

    +

    (inventory_forecast_risk["demand_category"]
        .isin(
            [
                "High Variability"
            ]
        )
        .astype(int)

    )

)


In [ ]:
priority_products = (

    inventory_forecast_risk

    [
        [
            "product_id",
            "category_x",
            "inventory_status",
            "days_of_inventory",
            "inventory_value",
            "forecast_risk",
            "demand_category",
            "risk_priority_score"
        ]
    ]

    .sort_values(
        "risk_priority_score",
        ascending=False
    )

    .head(20)

)


priority_products

,product_id,category_x,inventory_status,days_of_inventory,inventory_value,forecast_risk,demand_category,risk_priority_score
23907,PROD03908,Home & Kitchen,Overstock,245.00,"2,572.50",High Risk,High Variability,6
5228,PROD00229,Fashion,Stockout Risk,14.00,75.74,High Risk,High Variability,6
17377,PROD02378,Grocery,Stockout Risk,14.00,69.70,High Risk,High Variability,6
4092,PROD04093,Fashion,Overstock,588.00,"3,742.62",High Risk,High Variability,6
22178,PROD02179,Furniture,Overstock,285.60,"10,958.88",High Risk,High Variability,6
10228,PROD00229,Fashion,Overstock,401.80,"3,105.34",High Risk,High Variability,6
21600,PROD01601,Sports,Overstock,192.50,"1,412.40",High Risk,High Variability,6
8907,PROD03908,Home & Kitchen,Stockout Risk,5.60,58.80,High Risk,High Variability,6
3049,PROD03050,Beauty,Overstock,218.80,"1,400.00",High Risk,High Variability,6
16669,PROD01670,Office Supplies,Overstock,498.00,"2,372.97",High Risk,High Variability,6


In [ ]:
supplier_risk_exposure = (

    supplier_dashboard

    .groupby("supplier_risk")

    .agg(

        suppliers=(
            "supplier_id",
            "count"
        ),

        purchase_value=(
            "total_order_value",
            "sum"
        ),

        avg_score=(
            "supplier_performance_score",
            "mean"
        )

    )

    .sort_values(
        "purchase_value",
        ascending=False
    )

)


supplier_risk_exposure

,suppliers,purchase_value,avg_score
supplier_risk,,,
Medium Risk,88,"170,672,795.40",83.41
High Risk,84,"164,304,763.00",80.91
Low Risk,28,"64,485,263.85",94.65


In [ ]:
warehouse_risk_matrix = (

    warehouse_dashboard

    [
        [
            "warehouse_id",
            "warehouse_efficiency_score",
            "capacity_utilization_pct",
            "stockout_risk_skus",
            "overstock_skus"
        ]
    ]

)


warehouse_risk_matrix

,warehouse_id,warehouse_efficiency_score,capacity_utilization_pct,stockout_risk_skus,overstock_skus
0,WH001,90.50,86.77,502,494
1,WH002,77.30,85.89,502,526
2,WH003,94.80,86.35,491,500
3,WH004,69.10,86.96,479,504
4,WH005,81.50,85.92,492,512


In [ ]:
warehouse_risk_matrix["warehouse_risk_score"] = (

    (100 -
     warehouse_risk_matrix["warehouse_efficiency_score"])

    +

    (warehouse_risk_matrix["capacity_utilization_pct"]
     > 86.5)
     .astype(int)
     * 5

    +

    (warehouse_risk_matrix["stockout_risk_skus"]
     /
     warehouse_risk_matrix["stockout_risk_skus"].max()
    )

)


warehouse_risk_matrix.sort_values(
    "warehouse_risk_score",
    ascending=False
)

,warehouse_id,warehouse_efficiency_score,capacity_utilization_pct,stockout_risk_skus,overstock_skus,warehouse_risk_score
3,WH004,69.10,86.96,479,504,36.85
1,WH002,77.30,85.89,502,526,23.70
4,WH005,81.50,85.92,492,512,19.48
0,WH001,90.50,86.77,502,494,15.50
2,WH003,94.80,86.35,491,500,6.18


In [ ]:
# =====================================================
# Supply Chain Health Score
# =====================================================

inventory_health = (

    inventory_dashboard["inventory_status"]
    .isin(["Healthy"])
    .mean()
    *100

)


supplier_health = (

    supplier_dashboard["supplier_performance_score"]
    .mean()

)


warehouse_health = (

    warehouse_dashboard["warehouse_efficiency_score"]
    .mean()

)


forecast_health = (

    forecast_dashboard["avg_forecast_accuracy"]
    .mean()

)


supply_chain_health_score = (

    inventory_health * 0.40

    +

    supplier_health * 0.25

    +

    warehouse_health * 0.20

    +

    forecast_health * 0.15

)


supply_chain_health_score

np.float64(76.56325605714287)

In [ ]:
health_breakdown = pd.DataFrame({

    "Component":[
        "Inventory Health",
        "Supplier Performance",
        "Warehouse Efficiency",
        "Forecast Accuracy"
    ],

    "Score":[
        inventory_health,
        supplier_health,
        warehouse_health,
        forecast_health
    ],

    "Weight":[
        "40%",
        "25%",
        "20%",
        "15%"
    ]

})


health_breakdown

,Component,Score,Weight
0,Inventory Health,64.90,40%
1,Supplier Performance,83.94,25%
2,Warehouse Efficiency,82.64,20%
3,Forecast Accuracy,87.29,15%


# Executive Insights & Business Recommendations

## Overview

This exploratory analysis evaluated the overall supply chain performance across inventory management, supplier reliability, warehouse efficiency, and demand forecasting.

The analysis identified key operational risks, efficiency gaps, and improvement opportunities to support better inventory planning, supplier management, warehouse operations, and demand forecasting decisions.

---

# 1. Inventory Management Insights

## Key Findings

- The total inventory value is concentrated mainly within healthy inventory, indicating that most inventory is currently available and operationally usable.

- However, a significant portion of inventory value is tied to overstock and slow-moving products, creating potential risks of:
    - Excess working capital blockage
    - Increased storage costs
    - Product obsolescence

- Inventory velocity analysis shows a considerable number of slow-moving and dead-stock items with high days of inventory.

- Several products were identified with combined risks:
    - Overstock / Stockout status
    - High forecast risk
    - High demand variability

## Recommendations

- Reduce purchasing frequency for slow-moving and overstock products.
- Implement targeted promotional campaigns to clear excess inventory.
- Improve safety stock calculations for products with stockout risk.
- Use demand variability and forecast accuracy metrics while setting reorder points.
- Establish automated inventory alerts for high-risk SKUs.

---

# 2. Supplier Performance Insights

## Key Findings

- Supplier analysis shows that a large proportion of procurement value is associated with medium and high-risk suppliers.

- High-risk suppliers demonstrate:
    - Lower supplier performance scores
    - Lower delivery reliability
    - Higher defect rates

- Low-risk suppliers consistently achieve stronger delivery and quality performance.

## Recommendations

- Develop supplier improvement plans for medium and high-risk suppliers.
- Prioritize supplier reviews based on purchase value exposure.
- Negotiate improved delivery SLAs with underperforming suppliers.
- Develop backup supplier relationships for critical products.
- Increase monitoring frequency for suppliers with poor delivery performance.

---

# 3. Warehouse Performance Insights

## Key Findings

- Warehouse utilization levels are relatively consistent across locations, indicating balanced inventory distribution.

- However, warehouse efficiency scores show significant differences between facilities.

- WH004 represents the highest operational risk due to:
    - Lowest warehouse efficiency score
    - High utilization level
    - Inventory management inefficiencies

- WH003 demonstrates the strongest operational performance and can serve as a benchmark warehouse.

## Recommendations

- Investigate operational bottlenecks in low-performing warehouses.
- Analyze picking, storage, and inventory movement processes.
- Adopt best practices from high-performing warehouses.
- Optimize inventory placement based on SKU movement frequency.
- Improve warehouse workflow efficiency before increasing capacity.

---

# 4. Demand Forecasting Insights

## Key Findings

- Overall forecast accuracy is approximately 87%, indicating a reasonably reliable forecasting process.

- Forecast errors remain significant for certain products, especially those with:
    - High demand variability
    - Seasonal fluctuations
    - Promotion-driven demand

- High-risk forecast products show lower accuracy and higher uncertainty.

## Recommendations

- Improve forecasting models for highly variable products.
- Include additional demand drivers such as:
    - Promotions
    - Holidays
    - Seasonal patterns
    - Historical demand trends

- Review forecast performance regularly at product level.
- Apply different forecasting strategies for stable and volatile products.

---

# 5. Cross Functional Supply Chain Risk Insights

## Key Findings

The combined risk analysis identified products and operational areas where multiple issues overlap.

High-priority risks include:

- Products with excess inventory and poor forecast accuracy.
- Products with stockout risk and unpredictable demand.
- High procurement exposure from risky suppliers.
- Warehouses with lower operational efficiency.

These areas represent the highest opportunity for operational improvement.

## Recommendations

- Create a supply chain risk monitoring dashboard.
- Prioritize actions based on financial impact rather than individual metrics.
- Integrate inventory, supplier, warehouse, and forecasting data into a unified decision system.

---

# 6. Overall Supply Chain Health Assessment

## Supply Chain Health Score

The calculated overall supply chain health score is:

**76.56 / 100**

This indicates that the supply chain is performing at a good level but has improvement opportunities.

## Strengths

- Strong inventory availability
- Good supplier performance among top suppliers
- Balanced warehouse distribution
- Reasonable forecast accuracy

## Improvement Areas

- Reduce excess inventory exposure
- Improve supplier reliability
- Increase warehouse operational efficiency
- Improve forecasting for volatile products

---

# Final Business Conclusion

The analysis demonstrates that the organization has a stable supply chain foundation but can significantly improve operational efficiency through better inventory optimization, supplier risk management, warehouse process improvement, and demand forecasting enhancement.

By focusing on high-risk products, improving supplier collaboration, and adopting data-driven inventory planning strategies, the organization can reduce costs, improve service levels, and increase supply chain resilience.